## este codigo corre el codigo de "catalog_recommender.py" que usa clusters k means

## Entrenamos al modelo con el dataset

In [ ]:
import pandas as pd
from catalog_cluster_recommender import ClusterRecommender
import importlib, catalog_cluster_recommender as ccr
importlib.reload(ccr)
from catalog_cluster_recommender import ClusterRecommender

df = pd.read_csv("../data/articles.csv",dtype={"article_id": str, "product_code": str})

rec = ClusterRecommender(
    text_cols=["prod_name","detail_desc"],  # ajusta a las columnas que tengas
    cat_cols=["garment_group_name","section_name","index_name",
              "graphical_appearance_name","perceived_colour_master_name",
              "perceived_colour_value_name"],
    svd_components=128,                     # puedes subir/bajar si tienes más/menos datos
    random_state=42,
    text_weight=5.0,   # texto 3x más fuerte
    cat_weight=0.5
)
df["__text_all__"] = (df.get("prod_name","").fillna("").astype(str) + " " +
                      df.get("detail_desc","").fillna("").astype(str))
rec.fit(df)
#rec.save("agente_visual_cluster_with_weights.pkl")

## probamos el pkl

In [5]:
from catalog_cluster_recommender import ClusterRecommender
import pandas as pd

rec2 = ClusterRecommender.load("agente_visual_cluster_with_weights.pkl")

print(rec2.recommend_cart_ids(["0365102032"], k=10, mode="max"))


['0365102023', '0664891001', '0365102042', '0365102037', '0687407001', '0860302003', '0730066001', '0696778002', '0730066002', '0852724003']


## la funcion 

* recommend_for_cart si hay mas de un articulo, devuelve los articulos del cluster mas cercanos al alguno de los clusters (articulos del carrito)

In [15]:
# Top-10 similares para varios artículos (unión de clústeres)
top10_cart = rec.recommend_for_cart(["0822382003", "0816431002"], k=10, diversity=True)


print(top10_cart)

        article_id product_code                     prod_name  \
103118  0909529001      0909529       SPORT Harmony loose tee   
101112  0894457002      0894457                   MC Panamera   
101241  0895296001      0895296               ED French dress   
102900  0907563001      0907563  SPORT Bergamo quilted jacket   
101889  0900263001      0900263         LOGG Bixa terry dress   
84390   0815471002      0815471  SPORT Heaven shape HW tights   
102901  0907565001      0907565         SPORT Wildlife anorak   
82179   0807681006      0807681                       ED Mona   
99473   0884693002      0884693    Petite HM+ Apricot trouser   
77628   0790505004      0790505                MC Swift dress   

       garment_group_name section_name  index_name  \
103118       Jersey Fancy         H&M+  Ladieswear   
101112       Jersey Fancy         H&M+  Ladieswear   
101241       Jersey Fancy         H&M+  Ladieswear   
102900       Jersey Fancy         H&M+  Ladieswear   
101889       J

# la funcion 

* Si tu carrito tiene una playera y un iPad, calculamos el promedio de sus embeddings y regresamos el Top-10 más cercanos a ese centroide (o sea, “parecidos al conjunto” en promedio).


* avg regresa el centroide

* max regresa productos que al menos se parecen a uno de los productos 

* min regresa productos que mas se parecen a todos los productos recibidos

In [18]:

top10_cart_parecidos_a_ambos = rec.recommend_cart(["0365102032"], k=10, mode="avg")
top10_cart_parecidos_a_ambos.to_csv("../data/pruebas/avg_random/id_0365102032_textHasMoreWeight10x.csv",index=False)

## hagamos pruebas para avg 

## pruebas con los ids (inputs) aleatorio

In [6]:
import pandas as pd
import numpy as np
import os

ID_COL  = "article_id"
K       = 10
OUT_DIR = "../data/pruebas/avg_random_weights"
RANDOM_STATE = 123  # cambia si quieres otra aleatoriedad

def run_cart_tests_avg(rec, seeds_all, ks=(1,2,3,4), k=10, dentro_cluster=True, random_state=RANDOM_STATE):
    if getattr(rec, "_df", None) is None:
        raise RuntimeError("rec debe estar .fit o .load antes de correr pruebas.")

    os.makedirs(OUT_DIR, exist_ok=True)
    cat_cols = list(rec._df.columns)  # TODAS las columnas del catálogo
    meta_cols = ["__sim__", "rank", "is_seed", "origen", "ids_prueba", "error"]
    base_cols = cat_cols + meta_cols
    all_runs = []
    rng = np.random.default_rng(random_state)

    # Mantén solo IDs presentes en el modelo para poder muestrear sin reemplazo
    present = set(map(str, rec._article_index.keys()))
    pool = [s for s in map(str, seeds_all) if s in present]

    for n in ks:
        if len(pool) < n:
            print(f"[AVISO] faltan IDs válidos del modelo para n={n} (hay {len(pool)}).")
            continue

        # === ALEATORIO sin reemplazo para cada n ===
        seeds = rng.choice(pool, size=n, replace=False).tolist()
        ids_tag = "|".join(seeds)

        # ====== resultados ======
        try:
            df_out_raw = rec.recommend_cart(
                article_ids=seeds, k=k, mode="avg", dentro_cluster=dentro_cluster
            ).copy()
            err = np.nan
        except Exception as e:
            df_out_raw = pd.DataFrame({ID_COL: [], "__sim__": []})
            err = str(e)

        # Asegurar TODAS las columnas del catálogo en resultados (merge robusto)
        if not df_out_raw.empty:
            df_out = df_out_raw.merge(
                rec._df[cat_cols], on=ID_COL, how="left", suffixes=("", "_cat")
            )

            # Garantiza columna “limpia” para cada col del catálogo
            for c in cat_cols:
                if c not in df_out.columns:
                    sc = c + "_cat"
                    df_out[c] = df_out[sc] if sc in df_out.columns else np.nan

            # Elimina columnas _cat y ordena catálogo primero
            drop_suf = [c for c in df_out.columns if c.endswith("_cat")]
            if drop_suf:
                df_out = df_out.drop(columns=drop_suf)
            df_out = df_out[cat_cols + [c for c in df_out.columns if c not in cat_cols]]

            # rank y metadatos
            if "__sim__" in df_out.columns:
                df_out = df_out.sort_values("__sim__", ascending=False).reset_index(drop=True)
            df_out["rank"] = np.arange(1, len(df_out) + 1) if len(df_out) else np.nan
        else:
            df_out = pd.DataFrame(columns=base_cols)

        df_out["is_seed"]    = False
        df_out["origen"]     = "resultado"
        df_out["ids_prueba"] = ids_tag
        df_out["error"]      = err
        for c in meta_cols:
            if c not in df_out.columns:
                df_out[c] = np.nan
        df_out = df_out.reindex(columns=base_cols)

        # ====== seeds como FILAS de producto (con mismas columnas) ======
        if ID_COL not in rec._df.columns:
            raise RuntimeError(f"El catálogo no tiene columna {ID_COL}.")

        df_in = rec._df[rec._df[ID_COL].astype(str).isin(seeds)].copy()
        faltantes = [s for s in seeds if s not in set(df_in[ID_COL].astype(str))]
        if faltantes:
            df_in = pd.concat([
                df_in,
                pd.DataFrame([{**{c: np.nan for c in cat_cols}, ID_COL: s} for s in faltantes])
            ], ignore_index=True)

        df_in["__sim__"]     = np.nan
        df_in["rank"]        = 0
        df_in["is_seed"]     = True
        df_in["origen"]      = "input"
        df_in["ids_prueba"]  = ids_tag
        df_in["error"]       = np.nan
        df_in = df_in.reindex(columns=base_cols)

        # ====== unir y guardar ======
        run_df = pd.concat([df_in, df_out], ignore_index=True)
        safe_ids = "_".join(seeds)
        run_path = os.path.join(OUT_DIR, f"test_avg_{n}ids_{safe_ids}.csv")
        run_df.to_csv(run_path, index=False, encoding="utf-8")
        print(f"[OK] {run_path}")
        all_runs.append(run_df)

    if all_runs:
        master = pd.concat(all_runs, ignore_index=True)
        master.to_csv(os.path.join(OUT_DIR, "tests_avg_master.csv"), index=False, encoding="utf-8")
        print(f"[OK] Maestro: {os.path.join(OUT_DIR, 'tests_avg_master.csv')}")

# === cargar seeds desde articles.csv (preserva ceros a la izquierda)
df_articles = pd.read_csv("../data/articles.csv", dtype={ID_COL: str})
seed_pool = df_articles[ID_COL].dropna().astype(str).unique().tolist()

# correr (rec ya entrenado/cargado) — ahora elige IDs ALEATORIOS del CSV
run_cart_tests_avg(rec, seed_pool, ks=(1,2,3,4), k=K, dentro_cluster=True, random_state=RANDOM_STATE)

[OK] ../data/pruebas/avg_random_weights\test_avg_1ids_0365102032.csv
[OK] ../data/pruebas/avg_random_weights\test_avg_2ids_0736765013_0766402019.csv
[OK] ../data/pruebas/avg_random_weights\test_avg_3ids_0603774007_0618650001_0868393001.csv
[OK] ../data/pruebas/avg_random_weights\test_avg_4ids_0820719001_0579468005_0651558004_0689365007.csv
[OK] Maestro: ../data/pruebas/avg_random_weights\tests_avg_master.csv


## ahora hagamos el mismo codigo para pruebas pero con max 

In [7]:
import pandas as pd
import numpy as np
import os

ID_COL  = "article_id"
K       = 10
OUT_DIR = "../data/pruebas/max_random_weights"
RANDOM_STATE = 123  # cambia si quieres otra aleatoriedad

def run_cart_tests_avg(rec, seeds_all, ks=(1,2,3,4), k=10, dentro_cluster=True, random_state=RANDOM_STATE):
    if getattr(rec, "_df", None) is None:
        raise RuntimeError("rec debe estar .fit o .load antes de correr pruebas.")

    os.makedirs(OUT_DIR, exist_ok=True)
    cat_cols = list(rec._df.columns)  # TODAS las columnas del catálogo
    meta_cols = ["__sim__", "rank", "is_seed", "origen", "ids_prueba", "error"]
    all_runs = []
    rng = np.random.default_rng(random_state)

    present = set(map(str, rec._article_index.keys()))
    pool = [s for s in map(str, seeds_all) if s in present]

    for n in ks:
        if len(pool) < n:
            print(f"[AVISO] faltan IDs válidos del modelo para n={n} (hay {len(pool)}).")
            continue

        seeds = rng.choice(pool, size=n, replace=False).tolist()
        ids_tag = "|".join(seeds)

        # ====== resultados ======
        try:
            df_out_raw = rec.recommend_cart_v2(
                article_ids=seeds, k=k, mode="max", dentro_cluster=dentro_cluster, include_seed_sims=True
            ).copy()
            err = np.nan
        except Exception as e:
            df_out_raw = pd.DataFrame({ID_COL: [], "__sim__": []})
            err = str(e)

        if not df_out_raw.empty:
            df_out = df_out_raw.merge(
                rec._df[cat_cols], on=ID_COL, how="left", suffixes=("", "_cat")
            )
            for c in cat_cols:
                if c not in df_out.columns:
                    sc = c + "_cat"
                    df_out[c] = df_out[sc] if sc in df_out.columns else np.nan
            drop_suf = [c for c in df_out.columns if c.endswith("_cat")]
            if drop_suf:
                df_out = df_out.drop(columns=drop_suf)
            other_cols = [c for c in df_out.columns if c not in cat_cols]
            df_out = df_out[cat_cols + other_cols]
            if "__sim__" in df_out.columns:
                df_out = df_out.sort_values("__sim__", ascending=False).reset_index(drop=True)
            df_out["rank"] = np.arange(1, len(df_out) + 1) if len(df_out) else np.nan
        else:
            df_out = pd.DataFrame(columns=cat_cols + meta_cols)

        df_out["is_seed"]    = False
        df_out["origen"]     = "resultado"
        df_out["ids_prueba"] = ids_tag
        df_out["error"]      = err

        # columnas extra del recomendador (para conservarlas)
        extra_cols = [c for c in df_out.columns if c.startswith("best_seed") or c.startswith("sim_seed")]

        keep_cols_out = cat_cols + extra_cols + meta_cols
        for c in keep_cols_out:
            if c not in df_out.columns:
                df_out[c] = np.nan
        df_out = df_out[keep_cols_out]

        # ====== seeds como filas ======
        if ID_COL not in rec._df.columns:
            raise RuntimeError(f"El catálogo no tiene columna {ID_COL}.")
        df_in = rec._df[rec._df[ID_COL].astype(str).isin(seeds)].copy()
        faltantes = [s for s in seeds if s not in set(df_in[ID_COL].astype(str))]
        if faltantes:
            df_in = pd.concat([
                df_in,
                pd.DataFrame([{**{c: np.nan for c in cat_cols}, ID_COL: s} for s in faltantes])
            ], ignore_index=True)

        df_in["__sim__"]     = np.nan
        df_in["rank"]        = 0
        df_in["is_seed"]     = True
        df_in["origen"]      = "input"
        df_in["ids_prueba"]  = ids_tag
        df_in["error"]       = np.nan
        for c in extra_cols:
            if c not in df_in.columns:
                df_in[c] = np.nan
        keep_cols_in = cat_cols + extra_cols + meta_cols
        df_in = df_in.reindex(columns=keep_cols_in)

        # ====== unir y NORMALIZAR columnas únicas ======
        run_df = pd.concat([df_in, df_out], ignore_index=True)
        run_df = run_df.loc[:, ~run_df.columns.duplicated()]  # <- NUEVO: elimina columnas duplicadas

        # Guarda y acumula
        safe_ids = "_".join(seeds)
        run_path = os.path.join(OUT_DIR, f"test_max_random_{n}ids_{safe_ids}.csv")
        run_df.to_csv(run_path, index=False, encoding="utf-8")
        print(f"[OK] {run_path}")
        all_runs.append(run_df)

    if all_runs:
        # <- NUEVO: alinear todas las corridas al MISMO esquema y sin duplicados
        normalized = []
        for df in all_runs:
            df = df.loc[:, ~df.columns.duplicated()].copy()
            normalized.append(df)
        # Unión de columnas y reindex uniforme
        master_cols = list(dict.fromkeys(col for df in normalized for col in df.columns))
        normalized = [df.reindex(columns=master_cols) for df in normalized]

        master = pd.concat(normalized, ignore_index=True, sort=False)
        master_path = os.path.join(OUT_DIR, "tests_max_master_random.csv")
        master.to_csv(master_path, index=False, encoding="utf-8")
        print(f"[OK] Maestro: {master_path}")

# === cargar seeds desde articles.csv (preserva ceros a la izquierda)
df_articles = pd.read_csv("../data/articles.csv", dtype={ID_COL: str})
seed_pool = df_articles[ID_COL].dropna().astype(str).unique().tolist()

# correr (rec ya entrenado/cargado)
run_cart_tests_avg(rec, seed_pool, ks=(1,2,3,4), k=K, dentro_cluster=True, random_state=RANDOM_STATE)

[OK] ../data/pruebas/max_random_weights\test_max_random_1ids_0365102032.csv
[OK] ../data/pruebas/max_random_weights\test_max_random_2ids_0736765013_0766402019.csv
[OK] ../data/pruebas/max_random_weights\test_max_random_3ids_0603774007_0618650001_0868393001.csv
[OK] ../data/pruebas/max_random_weights\test_max_random_4ids_0820719001_0579468005_0651558004_0689365007.csv
[OK] Maestro: ../data/pruebas/max_random_weights\tests_max_master_random.csv


top10_cart.to_csv("../data/articles_including_their_cluster.csv",index=False)

top10_cart_parecidos_a_ambos.to_csv("../data/top10_cart_parecidos_a_ambos.csv",index=False)